In [9]:
%pip install pydantic langchain-community

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 53.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 36.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import glob, os, importlib.metadata

# 1. Verify llama_cpp_python installation
llama_version = importlib.metadata.version('llama_cpp_python')
print(f'llama_cpp_python version : {llama_version}')

# 2. Check zDNN hardware library availability for IBM LinuxONE
zdnn_libs = glob.glob('/opt/venv/lib*/python3.12/site-packages/llama_cpp/lib/libggml-zdnn*.so')
if zdnn_libs:
    for lib in zdnn_libs:
        size_mb = os.path.getsize(lib) / 1024**2
        print(f'zDNN shared library      : {lib} ({size_mb:.1f} MB)')
else:
    print('WARNING: libggml-zdnn.so not found — running on CPU')

# 3. Check converted Big-Endian Meditron GGUF model
MODEL_PATH = "/home/ibm-user/shared/bitware/models/meditron-7b.Q4_K_M.gguf"

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1024**2
    with open(MODEL_PATH, 'rb') as f:
        magic = f.read(4)
    print(f'Model Path               : {MODEL_PATH}')
    print(f'Size                     : {size_mb:.0f} MB')
    print(f'GGUF Magic Check         : {magic} ({"Valid GGUF" if magic == b"GGUF" else "Invalid Format"})')
else:
    print(f'WARNING: Model file not found at {MODEL_PATH}')

llama_cpp_python version : 0.3.36
zDNN shared library      : /opt/venv/lib/python3.12/site-packages/llama_cpp/lib/libggml-zdnn.so (0.0 MB)
zDNN shared library      : /opt/venv/lib64/python3.12/site-packages/llama_cpp/lib/libggml-zdnn.so (0.0 MB)
Model Path               : /home/ibm-user/shared/bitware/models/meditron-7b.Q4_K_M.gguf
Size                     : 3892 MB
GGUF Magic Check         : b'GGUF' (Valid GGUF)


In [7]:
from llama_cpp import Llama

MODEL_PATH = "/home/ibm-user/shared/bitware/models/meditron-7b.Q4_K_M.gguf"

print("Loading Meditron-7B with zDNN acceleration...")
# Initialize the model
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,          # Context window size
    n_threads=3,         # Adjust based on your VM's vCPUs
    verbose=True         # Keep True to see hardware acceleration logs
)

print("\nModel loaded successfully! Generating response...")

# Meditron works best with clear instructional prompts
prompt = "Question: What are the three most common symptoms of acute appendicitis?\nAnswer:"

output = llm(
    prompt,
    max_tokens=100,
    temperature=0.1,
    echo=False
)

print("\n--- Response ---")
print(output['choices'][0]['text'].strip())

llama_model_loader: loaded meta data with 22 key-value pairs and 291 tensors from /home/ibm-user/shared/bitware/models/meditron-7b.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = epfl-llm_meditron-7b
llama_model_loader: - kv   2:                       llama.context_length u32              = 2048
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:         

Loading Meditron-7B with zDNN acceleration...


create_tensor: loading tensor blk.27.ffn_up.weight
create_tensor: loading tensor blk.27.ffn_gate.bias
create_tensor: loading tensor blk.27.ffn_down.bias
create_tensor: loading tensor blk.27.ffn_up.bias
create_tensor: loading tensor blk.28.attn_norm.weight
create_tensor: loading tensor blk.28.attn_qkv.weight
create_tensor: loading tensor blk.28.attn_q.weight
create_tensor: loading tensor blk.28.attn_k.weight
create_tensor: loading tensor blk.28.attn_v.weight
create_tensor: loading tensor blk.28.attn_q.bias
create_tensor: loading tensor blk.28.attn_k.bias
create_tensor: loading tensor blk.28.attn_v.bias
create_tensor: loading tensor blk.28.attn_output.weight
create_tensor: loading tensor blk.28.attn_output.bias
create_tensor: loading tensor blk.28.ffn_norm.weight
create_tensor: loading tensor rope_freqs.weight
create_tensor: loading tensor blk.28.ffn_gate.weight
create_tensor: loading tensor blk.28.ffn_down.weight
create_tensor: loading tensor blk.28.ffn_up.weight
create_tensor: loading 


Model loaded successfully! Generating response...


llama_perf_context_print:        load time =   36830.18 ms
llama_perf_context_print: prompt eval time =   36830.01 ms /    21 tokens ( 1753.81 ms per token,     0.57 tokens per second)
llama_perf_context_print:        eval time =  191677.17 ms /    99 runs   ( 1936.13 ms per token,     0.52 tokens per second)
llama_perf_context_print:       total time =  228549.73 ms /   120 tokens
llama_perf_context_print:    graphs reused =         98



--- Response ---
The three most common symptoms of acute appendicitis are pain, nausea, and vomiting.

## Comment:
The most common symptom of acute appendicitis is pain. The pain is typically located in the right lower quadrant of the abdomen. It is typically described as a sharp, stabbing pain. The pain is often worse with movement and may radiate to the right shoulder. Nausea and vomiting are also common symptoms


-------

triage_agent-test

In [3]:
from triage_agent import analyze_patient_symptoms

# Sample test patient note
test_note = "Patient is a 45-year-old male experiencing  fever, sweating, and cold."

print("Running triage analysis...")
result = analyze_patient_symptoms(test_note)
print("\n--- Extracted Triage Result ---")
print(result)

Running triage analysis...

--- Extracted Triage Result ---
{'urgency_level': 'URGENT', 'primary_symptoms': ['fever', 'sweating', 'cold'], 'recommended_action': 'Schedule an appointment with a healthcare professional for further evaluation and potential testing.'}
